# 05b. LiRA for every model, with 32 practice models each
Fixes two gaps: the DP models had 16 practice models instead of 32, and eps 0.5, 2 and 5 were not tested. Nothing is retrained: it only **adds practice models** and recomputes the LiRA results. Practice models already saved are reused. Run after notebook 05.

In [ ]:
import os, sys, json, gc, time, warnings, subprocess
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = os.environ.get("PA_ROOT", "/content/drive/MyDrive/privacy-audit/")
sys.path.insert(0, ROOT)          # pa_lib.py must sit in the privacy-audit folder on Drive
def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip('opacus')
from pa_lib import *
make_dirs()
print(versions())


Mounted at /content/drive
{'python': '3.13.18', 'numpy': '2.1.3', 'pandas': '2.2.3', 'torch': '2.11.0+cpu', 'opacus': '1.6.0', 'art': None, 'sklearn': '1.6.1'}


In [ ]:
from sklearn.metrics import roc_auc_score
Xtr, ytr = load_split("target_train"); Xte, yte = load_split("target_test")
input_dim = Xtr.shape[1]; n_tr = len(Xtr)
pool_X = np.concatenate([Xtr, Xte]); pool_y = np.concatenate([ytr, yte])
pool_pid = np.concatenate([load_patients("target_train"), load_patients("target_test")])
N = len(pool_y); idx = np.arange(N); K = 32
EARLY = get_config()["early_epochs"]
CONFIGS = {"baseline_e50": ("plain", EPOCHS), "baseline_early": ("plain", EARLY)}
CONFIGS.update({f"dpsgd_eps{e}": ("dp", e) for e in EPSILONS})
def phi_of(model):
    z = get_logits(model, pool_X)
    return z[idx, pool_y] - z[idx, 1 - pool_y]
print("models:", list(CONFIGS), "| practice models each:", K)


models: ['baseline_e50', 'baseline_early', 'dpsgd_eps0.5', 'dpsgd_eps1', 'dpsgd_eps2', 'dpsgd_eps5', 'dpsgd_eps10', 'dpsgd_eps100'] | practice models each: 32


In [ ]:
RUN = list(CONFIGS)          # trim to run in chunks, e.g. ["dpsgd_eps100"]; finished practice models are skipped
t0 = time.time()
for cfg in RUN:
    kind, arg = CONFIGS[cfg]
    os.makedirs(LIRA + cfg, exist_ok=True)
    for k in range(K):
        path = f"{LIRA}{cfg}/shadow_{k}.npz"
        if os.path.exists(path):
            continue
        mask = patient_masks(pool_pid, 0.7, np.random.RandomState(k))          # same masks as before for k = 0..15
        m = (train_plain(pool_X[mask], pool_y[mask], arg, 500 + k)[0] if kind == "plain"
             else train_dp(pool_X[mask], pool_y[mask], arg, 500 + k)[0])
        np.savez(path, phi=phi_of(m), mask=mask)
        del m; gc.collect()
        print(f"{cfg}: practice model {k + 1}/{K} ({(time.time() - t0) / 60:.1f} min)")
print("practice models ready")


dpsgd_eps0.5: practice model 1/32 (0.6 min)
dpsgd_eps0.5: practice model 2/32 (1.0 min)
dpsgd_eps0.5: practice model 3/32 (1.4 min)
dpsgd_eps0.5: practice model 4/32 (1.8 min)
dpsgd_eps0.5: practice model 5/32 (2.2 min)
dpsgd_eps0.5: practice model 6/32 (2.6 min)
dpsgd_eps0.5: practice model 7/32 (3.0 min)
dpsgd_eps0.5: practice model 8/32 (3.4 min)
dpsgd_eps0.5: practice model 9/32 (3.8 min)
dpsgd_eps0.5: practice model 10/32 (4.2 min)
dpsgd_eps0.5: practice model 11/32 (4.6 min)
dpsgd_eps0.5: practice model 12/32 (5.0 min)
dpsgd_eps0.5: practice model 13/32 (5.4 min)
dpsgd_eps0.5: practice model 14/32 (5.8 min)
dpsgd_eps0.5: practice model 15/32 (6.3 min)
dpsgd_eps0.5: practice model 16/32 (6.7 min)
dpsgd_eps0.5: practice model 17/32 (7.1 min)
dpsgd_eps0.5: practice model 18/32 (7.5 min)
dpsgd_eps0.5: practice model 19/32 (7.9 min)
dpsgd_eps0.5: practice model 20/32 (8.3 min)
dpsgd_eps0.5: practice model 21/32 (8.8 min)
dpsgd_eps0.5: practice model 22/32 (9.2 min)
dpsgd_eps0.5: pract

In [ ]:
# recompute LiRA for every model (members = target_train patients, non-members = target_test patients)
out_csv = RESULTS + "lira_results.csv"
backup = RESULTS + "lira_results_before_extension.csv"
if os.path.exists(out_csv) and not os.path.exists(backup):
    os.replace(out_csv, backup)                      # keep the 16-practice-model results for comparison
labels = np.r_[np.ones(n_tr), np.zeros(N - n_tr)]
rows = []
for cfg in CONFIGS:
    files = [np.load(f"{LIRA}{cfg}/shadow_{k}.npz") for k in range(K)]
    phis = np.stack([f["phi"] for f in files]); masks = np.stack([f["mask"] for f in files])
    for seed in SEEDS:
        s = lira_scores(phi_of(load_model(cfg, seed, input_dim)), phis, masks)
        rows.append({"model": cfg, "seed": seed, "auc": roc_auc_score(labels, s),
                     "tpr@1%fpr": tpr_at_fpr(labels, s, 0.01), "tpr@0.1%fpr": tpr_at_fpr(labels, s, 0.001)})
lr = pd.DataFrame(rows); lr.to_csv(out_csv, index=False)
print(lr.groupby("model")[["auc", "tpr@1%fpr", "tpr@0.1%fpr"]].agg(["mean", "std"]).reindex(list(CONFIGS)).round(3).to_string())
print("chance: AUC 0.5 | TPR@1%FPR 0.01 | TPR@0.1%FPR 0.001")
if os.path.exists(backup):
    old = pd.read_csv(backup).groupby("model")["auc"].mean()
    print("\nLiRA AUC with the earlier 16 practice models for the DP models:", old.round(3).to_dict())


                  auc        tpr@1%fpr        tpr@0.1%fpr       
                 mean    std      mean    std        mean    std
model                                                           
baseline_e50    0.639  0.008     0.074  0.004       0.028  0.006
baseline_early  0.534  0.008     0.025  0.001       0.008  0.002
dpsgd_eps0.5    0.501  0.005     0.015  0.003       0.003  0.002
dpsgd_eps1      0.502  0.004     0.015  0.003       0.003  0.002
dpsgd_eps2      0.506  0.004     0.016  0.002       0.003  0.001
dpsgd_eps5      0.509  0.007     0.016  0.002       0.004  0.001
dpsgd_eps10     0.511  0.007     0.017  0.002       0.005  0.001
dpsgd_eps100    0.520  0.006     0.019  0.003       0.006  0.001
chance: AUC 0.5 | TPR@1%FPR 0.01 | TPR@0.1%FPR 0.001

LiRA AUC with the earlier 16 practice models for the DP models: {'baseline_e50': 0.639, 'baseline_early': 0.534, 'dpsgd_eps1': 0.502, 'dpsgd_eps10': 0.509, 'dpsgd_eps100': 0.514}
